<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day14_practice2_%EC%A0%84%EC%9D%B4%ED%95%99%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 전이학습
1. 손글씨(MNIST)로 CNN을 사전학습 ← 우리가 만든 거인
2. 사전 학습 한 그 가중치로 우리 데이터 옷(F-MNIST) 학습을 시작 :: 전이학습
3. 동결(freeze): 특징부는 잠그고(사전학습한건 그냥 두고) 분류기(헤드)만 학습
4. 파인튜닝 : 우리 데이터 옷(F-MNIST) 학습할 때 사전학습한 특징부도 다시 같이 학습

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [4]:
# 셀 1. 데이터 - 손글씨(사전학습용) / 옷(본 과제)
def load(ds_cls, root="./data"):
  tf = transforms.ToTensor()
  tr = ds_cls(root=root, train=True, download=True, transform=tf)
  te = ds_cls(root=root, train=False, download=True, transform=tf)
  return tr, te

mnist_tr, _ = load(datasets.MNIST)
fash_tr, fash_te = load(datasets.FashionMNIST)

small_fash = torch.utils.data.Subset(fash_tr, range(2000)) # Subset 부분 데이터셋
print(f"사전학습용 손글씨 {len(mnist_tr)}장 / 본 과제 옷 {len(small_fash)}장 (일부러 소량)") # 소량의 데이터로 전이학습

100%|██████████| 9.91M/9.91M [00:00<00:00, 46.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.06MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 7.93MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.79MB/s]
100%|██████████| 26.4M/26.4M [00:01<00:00, 17.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 275kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.98MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.8MB/s]


사전학습용 손글씨 60000장 / 본 과제 옷 2000장 (일부러 소량)


In [6]:
# 셀 2. 모델 CNN을 '특정부 + 분류기'로 분리
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential( # (백본backbone / 특징 추출기 / 인코더), 보는 법(전이될 부분), 사전학습
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 28*28 이미지크기 → 14*14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2) # 14*14 → 7*7
        )
        self.classifier = nn.Sequential( # (헤드head / 의사결정부 / 분류기), 판단(과제마다 교체)
            nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def train(model, dataset, epochs, lr=0.001):
    loader = DataLoader(dataset, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(42))
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(filter(lambda p : p.requires_grad, model.parameters()), lr=lr) # 동결(freeze)된 파라미터(requires_grad=False)는 제외, 기울기 계산 및 업데이트 여부
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        return model

def evaluate(model):
    loader = DataLoader(fash_te, batch_size=512)
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item() # 맞은 개수 누적
    return correct / len(fash_te) # 맞은 수 / 전체 = 정확도

In [7]:
# 셀 3. 사전학습 - 손글씨로 '보는 법'을 먼저 배운다
torch.manual_seed(42)
pretrained = CNN().to(device)
train(pretrained, mnist_tr, epochs=2)
torch.save(pretrained.state_dict(), "mnist_pretrained.pt") # state_dict() = 모델의 모든 가중치를 담은 딕셔너리, .pt 파일로 저장
print("사전학습 완료 → mnist_pretrained.pt 저장")

사전학습 완료 → mnist_pretrained.pt 저장


In [9]:
# 셀 4. 공정 대결 - 일반 vs 전이학습 (옷 2000장, 같은 1 epoch)

# 1. 일반 : 무작위 초기값에서 시작
torch.manual_seed(42)
scratch = CNN().to(device)
train(scratch, small_fash, epochs=1) # 옷 2000장으로 학습
acc_scratch = evaluate(scratch)

# 2. 전이학습: 사전학습한(손글씨 6만장) 가중치에서 시작 (분류기는 새로 만든다. 우리 데이터에 맞게)
torch.manual_seed(42)
transfer = CNN().to(device)
transfer.load_state_dict(torch.load("mnist_pretrained.pt")) # 사전학습 가중치 딕셔너리를 불러와서 현재 모델의 파라미터를 업데이트
transfer.classifier = nn.Sequential( # 분류기는 새것으로 교체, 우리 데이터에 맞게
    nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, 10).to(device)
)
train(transfer, small_fash, epochs=1)
acc_transfer = evaluate(transfer)

print(f"[옷 2000장 / 1 epoch 공정 대결]")
print(f"일반 시작 : {acc_scratch:.4f}")
print(f"손글씨 전이 시작 : {acc_transfer:.4f}")

[옷 2000장 / 1 epoch 공정 대결]
일반 시작 : 0.5444
손글씨 전이 시작 : 0.6816


In [10]:
# 셀 5. 동결(freeze) 방식의 전이학습 - 특징부는 잠그고 분류기만, 특징 훼손 방지 + 빠름, 데이터 아주 적을 때 사용
torch.manual_seed(42)
frozen = CNN().to(device)
frozen.load_state_dict(torch.load("mnist_pretrained.pt"))
for p in frozen.features.parameters():
    p.requires_grad = False # 특징부 동결, 기울기 계산과 업데이트가 되지 않는다. 역전파 때 갱신 안 됨(잠금)
frozen.classifier = nn.Sequential( # 분류기는 새것으로 교체, 우리 데이터에 맞게
    nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, 10).to(device)
)
train(frozen, small_fash, epochs=1)
acc_frozen = evaluate(frozen)

trainable = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
total = sum(p.numel() for p in frozen.parameters())
print(f"동결 방식의 전이학습(학습 파라미터 {trainable/1e3:.0f}K / 전체 {total/1e3:.0f}K)")
print(f"정확도: {acc_frozen:.4f}")

동결 방식의 전이학습(학습 파라미터 202K / 전체 207K)
정확도: 0.6939


In [14]:
# 셀 6. 파인튜닝(fine-tuning) 방식 - 백본(특징추출기)까지 전부 다시 학습한다
# resnet18 vs efficientnet_b0
# 데이터는 F-MNIST(1채널 28×28)를 ImageNet 모델 입력(3채널 224×224)에 맞춘다

_tf224 = transforms.Compose([transforms.Resize(224), transforms.ToTensor()]) # Compose: 전처리를 순서대로 묶음
_tr = Subset(datasets.FashionMNIST("./data", train=True, download=True, transform=_tf224), range(2000))
_te = Subset(datasets.FashionMNIST("./data", train=False, download=True, transform=_tf224), range(1000))
_trl = DataLoader(_tr, batch_size=32, shuffle=True)
_tel = DataLoader(_te, batch_size=64)

def to3ch(x):
  return x.repeat(1, 3, 1, 1) # 채널축만 3배로 복제 (B, 1, 224, 224) →(B, 3, 224, 224)

def build(name, n_classes=10): # 사전학습 모델 로드 → 마지막 Linear만 교체
  if name == "resnet18":
    m = models.resnet18(weights="IMAGENET1K_V1") # 백본: ImageNet 데이터셋으로 사전학습한 가중치째로 로드
    m.fc = nn.Linear(m.fc.in_features, n_classes) # 헤드(분류기)만 교체 (백본은 그대로 물려받음), m.fc.in_features 원래 입력 크기(512)
  else: # efficientnet_b0
    m = models.efficientnet_b0(weights="IMAGENET1K_V1")
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes) # 마지막 Linear만 내 클래스 수로 교체
  return m.to(device)

def train_eval_big(name, epochs=2, lr=0.01):
  torch.manual_seed(42)
  m = build(name)
  opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9) # momentum = 0.9: 이전 방향을 관성처럼 반영해 학습 안정/가속
  sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) # 학습률 스케쥴러. lr을 epochs에 걸쳐 코사인 감소
  loss_fn = nn.CrossEntropyLoss()
  # YOLO는 스케쥴러가 '자동'으로 켜져있다. YOLO 대신 전이학습(resnet18 등)을 쓰면 스케쥴러를 '직접' 넣어야 한다
  # 언제 사용 : 길게 학습할 때, SGD 계열일 때, 전이학습 미세조정(사전학습 가중치를 안 망치려 '낮게 시작→더 낮게')
  # CosineAnnealingLR: lr을 코사인 곡선을 따라 0으로 '매끄럽게' 낮춤
  # T_max=epochs은 전체 에폭에 걸쳐 한 번의 코사인 하강을 완성하라 → 초반엔 크게 배우고, 막판엔 lr이 0 근처라 미세하게 수렴

  for _ in range(epochs):
    m.train()
    for x, y in _trl:
      x, y = to3ch(x).to(device), y.to(device)
      loss = loss_fn(m(x), y)
      opt.zero_grad()
      loss.backward()
      opt.step()
    sch.step() # 에폭 끝마다 lr 조

  m.eval()
  correct = 0
  with torch.no_grad():
    for x, y in _tel:
      correct += (m(to3ch(x).to(device)).argmax(1) == y.to(device)).sum().item()
  n_params = sum(p.numel() for p in m.parameters())
  return correct / len(_te), n_params
print("[진짜 거인 전이학습 비교 - F-MNIST 2000장, 2에폭, SGD+코사인]")
for name in ["resnet18", "efficientnet_b0"]: # 두 백본을 차례로
  acc, n = train_eval_big(name)
  print(" " + name.ljust(16) + ": 정확도" + format(acc, ".4f") + "| 파라미터" + str(n//1000) + "K")
# 파인튜닝(finetune): 학습 파라미터 = 전체(백본까지 갱신), 보통 정확도 up, 대신 느리고 데이터 적으면 과적합/특징 훼손 위험
# 동결(freeze): 학습 파라미터 = 헤드뿐(전체의 0.1% 안팍), 빠르고 안정적, 데이터 아주 적을 때 유리, 대신 성능 상한은 파인튜닝보다 낮을 수 있음

[진짜 거인 전이학습 비교 - F-MNIST 2000장, 2에폭, SGD+코사인]
 resnet18        : 정확도0.8810| 파라미터11181K
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 75.9MB/s]


 efficientnet_b0 : 정확도0.8580| 파라미터4020K
